# 02 – Load and validate

Loads the extracted games into DuckDB and checks the data before analysis.
Each check records what was tested, what was found, and the cleaning decision.

**Requires:** `data/processed/games.csv` (created by `src/extract_headers.py`).
**Run:** top to bottom. Paths are relative to the project root.

## Setup and load

In [1]:
import os
import duckdb

# Work from the project root so every path matches the script and the SQL files.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

database_connection = duckdb.connect("data/lichess.duckdb")

/Users/alec/lichess/lichess-ratings


## 01_load.sql

In [ ]:
database_connection.execute("DROP TABLE IF EXISTS games_raw")

with open("sql/01_load.sql") as sql_file:
    database_connection.execute(sql_file.read())

database_connection.sql("SELECT COUNT(*) AS total_rows FROM games_raw").show()

In [ ]:
database_connection.sql("""
SELECT * FROM games_raw LIMIT 5;
""").show()

In [ ]:
database_connection.sql("""
DESCRIBE games_raw;
""").show()

## Check 1: Duplicate games

In [ ]:
database_connection.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(Site) AS total_sites,
    COUNT(DISTINCT Site) AS distinct_sites
FROM games_raw;
""").show()

Site is each game’s URL and should be unique. All three counts are 4,770,357: every game has an ID and none repeat. 0 games removed.

## Check 2: Result values

In [ ]:
database_connection.sql("""
SELECT
    Result,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_games
FROM games_raw
GROUP BY Result
ORDER BY games DESC;
""").show()

Exactly three values, all scorable: White wins 49.74%, Black wins 46.61%, draws 3.64%. No * (unfinished) games. 0 removed. White’s unadjusted edge is examined in the analysis.

## Check 3: Termination types

In [ ]:
database_connection.sql("""
SELECT
    Termination,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS percent_of_games
FROM games_raw
GROUP BY Termination
ORDER BY games DESC;
""").show()

Checked percent results and total results match. Found Rules infraction type for Termination and will exclude Abandoned and Rules infraction (20,373 games) as the outcome of these games wasn’t decided by play.  *May not sum to 100 due to rounding*

## Check 4: Ratings

### 4a: Unusable ratings

In [ ]:
database_connection.sql("""

SELECT 
    SUM(CASE WHEN TRY_CAST(WhiteElo AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS white_elo_try_cast_null,
    SUM(CASE WHEN WhiteElo IS NULL THEN 1 ELSE 0 END) AS white_elo_null,
    SUM(CASE WHEN WhiteElo = '?' THEN 1 ELSE 0 END) AS white_elo_question_mark,
    SUM(CASE WHEN TRY_CAST(BlackElo AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS black_elo_try_cast_null,
    SUM(CASE WHEN BlackElo IS NULL THEN 1 ELSE 0 END) AS black_elo_null,
    SUM(CASE WHEN BlackElo = '?' THEN 1 ELSE 0 END) AS black_elo_question_mark,
    SUM(CASE WHEN BlackElo = '?' OR WhiteElo = '?' THEN 1 ELSE 0 END) AS games_with_unusable_rating
FROM games_raw

""").show()


Converted `WhiteElo` and `BlackElo` to integers with `TRY_CAST`.
11 White and 26 Black ratings fail to convert, and every one of them is `'?'` (no `NULL`s, no other
non-numeric values). 37 games have at least one unusable rating; no game has both.
**Decision:** exclude these 37 games (0.0008% of the month), because the Elo formula needs both
players' ratings.

### 4b: Rating range

In [ ]:
database_connection.sql("""

SELECT
    MIN(TRY_CAST(WhiteElo AS INTEGER)) AS min_white_elo,
    MAX(TRY_CAST(WhiteElo AS INTEGER)) AS max_white_elo,
    MIN(TRY_CAST(BlackElo AS INTEGER)) AS min_black_elo,
    MAX(TRY_CAST(BlackElo AS INTEGER)) AS max_black_elo
FROM games_raw

""").show()

Find the lowest and highest rating of the month to check if they're plausible. Absurd value (ex 0 or 9000) could distort chart and analysis.
**Finding:** No ratings are an absurd amount.  min_white_elo=739, min_black_elo=740, max_white_elo=2998, max_black_elo=2995. 0 removed

### 4c: The 1500 starting rating

In [ ]:
database_connection.sql("""

SELECT
    TRY_CAST(WhiteElo AS INTEGER) AS white_rating,
    COUNT(*) AS games
FROM games_raw
WHERE white_rating BETWEEN 1490 AND 1510
GROUP BY white_rating
ORDER BY white_rating


""").show()

database_connection.sql("""

SELECT
    SUM(CASE WHEN TRY_CAST(WhiteElo AS INTEGER) = 1500 OR TRY_CAST(BlackElo AS INTEGER) = 1500 THEN 1 ELSE 0 END) AS games_with_either_player_1500_elo
FROM games_raw

""").show()

Test whether 1500, Lichess's starting rating, is unusually common. Accounts started are given 1500 as an initial placeholder before their first game.
**Finding:** The 1500 rating has 40540 games, about 8 times more than all the other Elo values 1490-1510 which are around 5000 games. 70,899 games (1.5%) have at least one player at 1500.
**Decision:** Keep the 1500 games, but flag them. Add a true/false column in clean table then run the main calibration both ways, with and without flagged games. Decided to go this route instead of deleting all 1500 rated games so its effect can be measured with a sensitivity analysis. Might be an important takeaway from this project. 
**Limitation:** The flag only catches each player’s first game in a time control, so the true effect of unreliable ratings is likely larger. Lichess keeps a separate rating per time control, so 1500 also marks experienced players new to a speed.

## Check 5: Time controls

### 5a: Time control values

In [ ]:
database_connection.sql("""

SELECT 
    COUNT(DISTINCT TimeControl) AS distinct_time_control
FROM games_raw

""").show()

database_connection.sql("""

SELECT
    TimeControl,
    COUNT(*) AS games
FROM games_raw
GROUP BY TimeControl
ORDER BY games DESC
LIMIT 15

""").show()

### 5b: Parsing base and increment

In [ ]:
database_connection.sql("""

SELECT
    SUM(CASE WHEN TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS base_is_null,
    SUM(CASE WHEN TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS increment_is_null,
    SUM(CASE WHEN TimeControl = '-' THEN 1 ELSE 0 END) AS time_control_is_dash
    
FROM games_raw

""").show()

`TimeControl` is stored as `base+increment` in seconds
(e.g. `300+3`). There are 856 distinct values. The most common are `60+0` (882,796 games),
`300+0` (661,672), and `180+0` (613,479), followed by a long tail of custom clocks.
Split each value on `+` with `split_part` and converted both parts with `TRY_CAST`.
20,924 games (0.44%) fail to parse, and every one of them is `-` (no clock, which means
correspondence chess). **Decision:** keep these games but label them "Correspondence"
in the speed category (5c), rather than forcing them into a clock-based category.
Correspondence uses different rules (opening books allowed) and its own rating pool,
so it will be analyzed separately from timed games.

### 5c: Speed categories

In [ ]:
database_connection.sql("""

WITH parsed_time_controls AS (
    SELECT
        TimeControl,
        Event,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) AS base_seconds,
        TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER) AS increment_seconds,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) + 40*(TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER)) AS estimated_duration_seconds
    FROM games_raw
)
SELECT 
    CASE 
        WHEN base_seconds IS NULL THEN 'Correspondence'
        WHEN estimated_duration_seconds <= 29 THEN 'UltraBullet'
        WHEN estimated_duration_seconds <= 179 THEN 'Bullet'
        WHEN estimated_duration_seconds <= 479 THEN 'Blitz'
        WHEN estimated_duration_seconds <= 1499 THEN 'Rapid'
        WHEN estimated_duration_seconds >= 1500 THEN 'Classical'
        ELSE 'Unknown'
    END AS speed_category,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_games
FROM parsed_time_controls
GROUP BY speed_category
ORDER BY games DESC;


""").show()

In [ ]:
database_connection.sql("""

WITH parsed_time_controls AS (
    SELECT
        TimeControl,
        Event,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) AS base_seconds,
        TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER) AS increment_seconds,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) + 40*(TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER)) AS estimated_duration_seconds
    FROM games_raw
)
SELECT
    split_part(Event, ' ', 2) AS event_label_2016,
    CASE 
        WHEN base_seconds IS NULL THEN 'Correspondence'
        WHEN estimated_duration_seconds <= 29 THEN 'UltraBullet'
        WHEN estimated_duration_seconds <= 179 THEN 'Bullet'
        WHEN estimated_duration_seconds <= 479 THEN 'Blitz'
        WHEN estimated_duration_seconds <= 1499 THEN 'Rapid'
        WHEN estimated_duration_seconds >= 1500 THEN 'Classical'
        ELSE 'Unknown'
    END AS speed_category,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_games
FROM parsed_time_controls
GROUP BY speed_category, event_label_2016
ORDER BY speed_category, event_label_2016 DESC;


""").show()

As of 10/14/26, Lichess FAQ states that "Lichess time controls are based on estimated game duration = (clock initial time in seconds) + 40 × (clock increment)." They are sorted into categories:
- ≤ 29s = UltraBullet
- ≤ 179s = Bullet
- ≤ 479s = Blitz
- ≤ 1499s = Rapid
- ≥ 1500s = Classical

Since the data I am using is from 2016, none of the event labels are Rapid because this was not its own category in 2016. This results in all of the games that would be considered Rapid in 2026 to be grouped under the event name Classical in 2016. 93% of games labeled “Classical” in 2016 (1,143,800 of 1,230,129) are Rapid by today’s definitions. Additionally, no games in this month qualify as UltraBullet. The games with a - time control, and the 2016 “Correspondence” label match exactly (20,924 each), which confirms the previous decision to keep in separate category.

**Findings**
- Blitz: 1,896,675 games (39.76%)
- Bullet: 1,622,629 games (34.01%)
- Rapid: 1,143,800 games (23.98%)
- Classical: 86,329 games (1.81%)
- Correspondence: 20,924 games (0.44%)
  
**Decision** Use the modern categories, derived from TimeControl, rather than the Event labels. Why: consistent with Lichess’s current definitions, and it separates Rapid from Classical, which the 2016 labels merged.

## Check 6: Date range

### 6a: Time window

In [ ]:
database_connection.sql("""

WITH game_times AS (
    SELECT
        UTCDate,
        UTCTime,
        try_strptime(UTCDate || ' ' || UTCTime, '%Y.%m.%d %H:%M:%S') AS played_at
    FROM games_raw
)
SELECT
    MIN(played_at) AS earliest_played_at,
    MAX(played_at) AS latest_played_at,
    SUM(CASE WHEN played_at IS NULL THEN 1 ELSE 0 END) AS null_played_at,
    SUM(CASE WHEN played_at >= TIMESTAMP '2016-01-31 23:00:00' THEN 1 ELSE 0 END) AS games_at_or_after_cutoff
FROM game_times
""").show()

Combined UTCDate and UTCTime into a timestamp and tested whether the file spans the appropriate 23:00 UTC Dec 31 to 23:00 UTC Jan 31. **Findings** Earliest is 2015-12-31 23:00:00, latest 2016-01-31 22:59:59, 0 failed conversions, 0 games outside the window. 0 games to remove. Additionally, the first and last rows of the file (23:00:03 and 22:59:44) aren’t the earliest and latest games, because the file isn’t in strict time order.

### 6b: Games dated Dec 31

In [ ]:
database_connection.sql("""

WITH game_times AS (
    SELECT
        UTCDate,
        UTCTime,
        try_strptime(UTCDate || ' ' || UTCTime, '%Y.%m.%d %H:%M:%S') AS played_at
    FROM games_raw
)
SELECT
    hour(played_at) AS hour_played_at,
    COUNT(*) AS games
FROM game_times
WHERE UTCDate = '2015.12.31'
GROUP BY hour_played_at
""").show()


Check number of games on Dec 31, 5,124 games (0.11%), all in the 23:00 hour (implied by 6a). This is consistent with Lichess cutting monthly files at midnight Central European Time (UTC+1 in winter). **Decision** Keep all games, 0 removed.

## Check 7: Openings

### 7a: Missing and invalid labels

In [ ]:
database_connection.sql("""

SELECT 
    SUM(CASE WHEN Opening = '?' THEN 1 ELSE 0 END) AS opening_question_mark,
    SUM(CASE WHEN ECO = '?' THEN 1 ELSE 0 END) AS eco_question_mark,
    SUM(CASE WHEN Opening = '?' AND ECO = '?' THEN 1 ELSE 0 END) AS opening_and_eco_question_mark,
    SUM(CASE WHEN ECO <> '?' AND NOT regexp_matches(ECO, '^[A-E][0-9]{2}$') THEN 1 ELSE 0 END) AS eco_not_question_and_not_valid_pattern,
    SUM(CASE WHEN Opening = '?' AND Termination = 'Abandoned' THEN 1 ELSE 0 END) AS opening_question_mark_and_abandoned
FROM games_raw
""").show()   

Opening and ECO are missing '?' in 14496 games and are always together. 0 ECO codes have an invalid format. 11,521 of the 14496 are abandoned games, which are already excluded in check 3, so 2,975 remain. **Decision:** keep them in the clean table as calibration doesn't use the openings. They will only be excluded from the opening analysis.

### 7b: Opening families

#### Step 1: First pass (split on the colon)

In [ ]:
database_connection.sql("""

WITH opening_clean AS (
    SELECT
        Opening,
        split_part(Opening, ':', 1) AS opening_family
    FROM games_raw
)
SELECT
    COUNT(DISTINCT Opening) AS distinct_openings,
    COUNT(DISTINCT opening_family) AS distinct_opening_family
FROM opening_clean
WHERE Opening <> '?'
""").show()  

database_connection.sql("""

WITH opening_clean AS (
    SELECT
        Opening,
        split_part(Opening, ':', 1) AS opening_family
    FROM games_raw
)
SELECT
    opening_family,
    COUNT(*) as games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_large_families
FROM opening_clean
WHERE Opening <> '?'
GROUP BY opening_family
HAVING COUNT(*) >= 1000
ORDER BY games DESC
LIMIT 15
""").show()  

Took the family as the part before the first colon: 2,901 openings and 349 families. The top 15 look reasonable, but the long tail hides problem addressed in Step 2.

#### Step 2: Measure the old naming patterns

In [ ]:
database_connection.sql("""

WITH opening_clean AS (
    SELECT
        Opening,
        split_part(Opening, ':', 1) AS opening_family
    FROM games_raw
)
SELECT
    COUNT(*) as games,
    SUM(CASE WHEN opening_family LIKE '%#%' THEN 1 ELSE 0 END) AS games_family_has_hash,
    SUM(CASE WHEN opening_family LIKE '%,%' THEN 1 ELSE 0 END) AS games_family_has_comma
FROM opening_clean
WHERE Opening <> '?'



""").show()  

2016 names use older conventions: numbered suffixes (#) in 330802 games (7.0%), and commas instead of colons in 96355 games (2.0%).
These split real families into fragments.

#### Step 3: Inspect examples

In [ ]:
database_connection.sql("""

WITH opening_clean AS (
    SELECT
        Opening,
        split_part(Opening, ':', 1) AS opening_family
    FROM games_raw
)
SELECT
    opening_family,
    COUNT(*) as games
FROM opening_clean
WHERE Opening <> '?' AND opening_family LIKE '%,%'
GROUP BY opening_family
ORDER BY games DESC
LIMIT 20

""").show()  

database_connection.sql("""

WITH opening_clean AS (
    SELECT
        Opening,
        split_part(Opening, ':', 1) AS opening_family
    FROM games_raw
)
SELECT
    opening_family,
    COUNT(*) as games
FROM opening_clean
WHERE Opening <> '?' AND opening_family LIKE '%#%'
GROUP BY opening_family
ORDER BY games DESC
LIMIT 20

""").show()  

'#' always appears at the end as '#number' (e.g Pirc Defense #4), so stripping with regex is safe. The text before the first comma is always a sensible family (e.g. King's Gambit Accepted, Fischer Defense → King's Gambit Accepted). **Decision:** King’s Gambit Accepted, King’s Gambit Declined, and King’s Gambit stay separate families, consistent with Queen’s Gambit Declined. Remove the part after the first comma.

#### Step 4: Normalize and verify

In [ ]:
database_connection.sql("""

WITH opening_families AS (
    SELECT
        Opening,
        regexp_replace(split_part(split_part(Opening, ':', 1), ',', 1), ' #[0-9]+$', '') AS opening_family
    FROM games_raw
)
SELECT
    COUNT(*) as games,
    SUM(CASE WHEN opening_family LIKE '%#%' THEN 1 ELSE 0 END) AS games_family_has_hash,
    SUM(CASE WHEN opening_family LIKE '%,%' THEN 1 ELSE 0 END) AS games_family_has_comma
FROM opening_families
WHERE Opening <> '?'

""").show()  

database_connection.sql("""

WITH opening_families AS (
    SELECT
        Opening,
        regexp_replace(split_part(split_part(Opening, ':', 1), ',', 1), ' #[0-9]+$', '') AS opening_family
    FROM games_raw
)
SELECT
    COUNT(DISTINCT Opening) AS distinct_openings,
    COUNT(DISTINCT opening_family) AS distinct_opening_family
FROM opening_families
WHERE Opening <> '?'
""").show()  

database_connection.sql("""

WITH opening_families AS (
    SELECT
        Opening,
        regexp_replace(split_part(split_part(Opening, ':', 1), ',', 1), ' #[0-9]+$', '') AS opening_family
    FROM games_raw
)
SELECT
    opening_family,
    COUNT(*) as games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_large_families
FROM opening_families
WHERE Opening <> '?'
GROUP BY opening_family
HAVING COUNT(*) >= 1000
ORDER BY games DESC
LIMIT 15
""").show()  

Fix: cut at the colon, then the comma, then strip '#' with regexp_replace. Successful: 0 leftover '#' or commas and reduced families from 349 to 168. Merges add up exactly and previous openings like the Philidor Defense which was previously fragmented moved up to 8th largest. (143581 games) **Decision:** use this cleaned opening_family in the clean table. For opening analysis require at least 1,000 games per family.